In [ ]:
import pandas as pd
from pathlib import Path

path = Path.cwd() / "task_2_data_ex.xlsx"

data = pd.read_excel(path)
display(data.head())



## ACTUAL SOLUTION


In [18]:
def get_produced_materials(data: pd.DataFrame) -> set:
    """Set of all material IDs that appear as a produced material."""
    return set(data["produced_material"].unique())


def get_fin_materials(data: pd.DataFrame):
    """IDs of all FIN materials."""
    return data.loc[data["produced_material_release_type"] == "FIN", "produced_material"].unique()


def get_material_data(data: pd.DataFrame, material_id: int) -> pd.DataFrame:
    """Returns all data of the selected produced_material (based on "produced_material" column)"""
    return data.loc[data["produced_material"] == material_id]


def explode_material_tree(data: pd.DataFrame, material_id: int, produced_materials: set,
                          visited: set | None = None) -> pd.DataFrame:
    """
    Top-down walk from material_id: returns every material -> component row in its
    sub-tree. `visited` accumulates across the walk to prevent cycles.
    """
    if visited is None:
        visited = set()
    if material_id in visited:
        return data.iloc[0:0]
    visited.add(material_id)

    material_data = get_material_data(data, material_id)
    frames = [material_data]
    for comp in material_data["component_material"].unique():
        if comp in produced_materials: # if a base component - do not explore (nothing to explore)
            frames.append(explode_material_tree(data, int(comp), produced_materials, visited))
    return pd.concat(frames, ignore_index=True)


def explode_fin(data: pd.DataFrame, fin_id: int, produced_materials: set) -> pd.DataFrame:
    """
    All material -> component rows in the tree under a FIN, tagged with an ID of that FIN.
    Starts below the FIN, at its PROD sub-tree, so the FIN itself is not a material row.
    """
    visited: set = set()
    frames = []
    for sub in get_material_data(data, fin_id)["component_material"].unique():
        if sub in produced_materials: # if a base component - do not explore (nothing to explore)
            frames.append(explode_material_tree(data, int(sub), produced_materials, visited))
    if not frames:
        return data.iloc[0:0]
    return pd.concat(frames, ignore_index=True).assign(_fin_id=fin_id)


def get_fin_attributes(data: pd.DataFrame, fin_ids) -> pd.DataFrame:
    """
    FIN attributes per plant/month, renamed to the fin_* output columns.
    """
    columns = ["plant_id", "year", "month", "produced_material",
               "produced_material_production_type", "produced_material_release_type",
               "produced_material_quantity"]
    
    return (data.loc[data["produced_material"].isin(fin_ids), columns].drop_duplicates()
                .rename(columns={
                    "produced_material":                 "fin_material_id",
                    "produced_material_release_type":    "fin_material_release_type",
                    "produced_material_production_type": "fin_material_production_type",
                    "produced_material_quantity":        "fin_production_quantity",
                }))


def sum_quantities_yearly(dataframe_with_fin: pd.DataFrame) -> pd.DataFrame:
    """
    Rename the explosion to the output schema and sum the three quantities per
    plant, year, FIN, produced material and component.
    """
    renamed = dataframe_with_fin.rename(columns={
        "plant_id":                          "plant",
        "produced_material":                 "prod_material_id",
        "produced_material_release_type":    "prod_material_release_type",
        "produced_material_production_type": "prod_material_production_type",
        "produced_material_quantity":        "prod_material_production_quantity",
        "component_material":                "component_id",
        "component_material_release_type":   "component_material_release_type",
        "component_material_production_type":"component_material_production_type",
        "component_material_quantity":       "component_consumption_quantity",
    })

    group_cols = [
        "plant", "year",
        "fin_material_id", "fin_material_release_type", "fin_material_production_type",
        "prod_material_id", "prod_material_release_type", "prod_material_production_type",
        "component_id", "component_material_release_type", "component_material_production_type",
    ]

    # dropna=False - ADD/RM components have NaN production_type
    # without this those component rows would be silently dropped.
    summary = renamed.groupby(group_cols, as_index=False, dropna=False).agg(
        fin_production_quantity=("fin_production_quantity", "sum"),
        prod_material_production_quantity=("prod_material_production_quantity", "sum"),
        component_consumption_quantity=("component_consumption_quantity", "sum"),
    )

    column_order = [
        "plant",
        "fin_material_id", "fin_material_release_type", "fin_material_production_type", "fin_production_quantity",
        "prod_material_id", "prod_material_release_type", "prod_material_production_type", "prod_material_production_quantity",
        "component_id", "component_material_release_type", "component_material_production_type", "component_consumption_quantity",
        "year"
    ]
    return summary[column_order]


def build_bom_explosion(data: pd.DataFrame) -> pd.DataFrame:
    """
    Full top-down BoM explosion for every FIN material, per plant and year.
    Each row is one material -> component relationship in a FIN's production tree;
    shared sub-assemblies appear under every FIN that uses them.
    """
    produced_materials = get_produced_materials(data)
    fin_ids = get_fin_materials(data)

    exploded = pd.concat([explode_fin(data, int(f), produced_materials) for f in fin_ids],
                         ignore_index=True)

    fin_info = get_fin_attributes(data, fin_ids)
    merged = exploded.merge(
        fin_info,
        left_on=["plant_id", "year", "month", "_fin_id"],
        right_on=["plant_id", "year", "month", "fin_material_id"],
        how="inner",
    )
    return sum_quantities_yearly(merged)


final = build_bom_explosion(data)
display(final)

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year
0,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0,2024
1,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2024
2,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2024
3,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,70000,RM,NaN,30498.0,2024
4,RLT_10,10000,FIN,8002,11708.0,80000,PROD,8000,23478.0,90005,ADD,NaN,829.0,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,RLT_16,10008,FIN,8002,12067.0,80018,PROD,8001,21410.0,80008,PROD,8000.0,23516.0,2024
96,RLT_16,10008,FIN,8002,12067.0,80018,PROD,8001,21410.0,90044,ADD,NaN,1198.0,2024
97,RLT_16,10008,FIN,8002,12067.0,80078,PROD,8007,10676.0,80018,PROD,8001.0,41471.0,2024
98,RLT_16,10008,FIN,8002,12067.0,80078,PROD,8007,10676.0,90042,ADD,NaN,355.0,2024
